# Silver Transactions (`pipelines/02_silver_transactions.ipynb`)

**Reads:** `workspace.sentinel_bronze.transactions`
**Writes:** `workspace.sentinel_silver.transactions` (accepted rows) and `workspace.sentinel_silver.quarantine` (rejected or extra duplicate rows)

**Flow:** `read → clean & check fields → find duplicate IDs → route dispositions → save → verify counts`

Every Bronze row ends in exactly one of three places:

| `record_disposition` | Meaning | Goes to |
| --- | --- | --- |
| `accepted` | Valid row, first copy of its ID | `silver.transactions` |
| `duplicate_identical` | Valid row, extra copy of an ID with identical business content | `silver.quarantine` |
| `quarantined` | Failed a field check, or its ID has conflicting copies | `silver.quarantine` |

In [ ]:
# Configuration: table names, column lists, and ordinary numeric amount pattern
from datetime import date, datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window

BRONZE_TABLE = "workspace.sentinel_bronze.transactions"
SILVER_SCHEMA = "workspace.sentinel_silver"
SILVER_TRANSACTIONS_TABLE = f"{SILVER_SCHEMA}.transactions"
SILVER_QUARANTINE_TABLE = f"{SILVER_SCHEMA}.quarantine"
MANIFEST_PATH = "/Volumes/workspace/sentinel_sources/replay_staging/replay_v1/manifest"

# The 21 source business columns. Two copies of a transaction ID are "identical"
# only when all 21 match. File and ingestion metadata columns are not compared.
BUSINESS_COLUMNS = [
    "amt", "category", "cc_num", "city", "city_pop", "dob", "first", "gender",
    "job", "last", "lat", "long", "merch_lat", "merch_long", "merchant", "state",
    "street", "trans_date_trans_time", "trans_num", "unix_time", "zip",
]

# Accepted Silver keeps no personal cardholder attributes.
ACCEPTED_COLUMNS = [
    "trans_num", "cc_num", "merchant", "category", "amount", "event_time",
    "replay_batch", "source_file", "source_row_index",
    "_ingested_file_path", "_ingestion_timestamp",
]

QUARANTINE_COLUMNS = [
    "record_disposition", "duplicate_status", "duplicate_rank", "validation_reasons",
    "trans_num", "cc_num", "merchant", "category", "amount", "event_time",
    "raw_trans_num", "raw_cc_num", "raw_merchant", "raw_amt",
    "raw_trans_date_trans_time", "unix_time", "_rescued_data",
    "replay_batch", "source_file", "source_row_index",
    "_ingested_file_path", "_ingestion_timestamp",
]

# Simpler amount policy: ordinary numeric text (optional minus sign, digits, and at most 2 decimals).
# Examples accepted by pattern: "12", "12.5", "12.50", "-15.25".
# Examples rejected as invalid_amount: "abc", "19.999" (>2 decimals), "+5", ".50", "1e3".
AMOUNT_PATTERN = r"^-?[0-9]+(\.[0-9]{1,2})?$"

## Step 1: Clean the text and check every field

Bronze stores source values as text. This function:
1. Keeps the original text of key columns as `raw_*` for quarantine inspection.
2. Trims spaces and turns empty strings into `NULL` (`F.nullif(F.trim(col), F.lit(""))`).
3. Converts `event_time` (`TIMESTAMP_NTZ`) and `amount` (`DECIMAL(18, 2)`) with `try_cast`, which returns `NULL` instead of raising an error when text is malformed.
4. Builds `field_reasons` using `F.array_compact(F.array(...))`, which collects all matching failure reasons and removes `NULL` entries without needing a `lambda`.

**Simpler amount policy & examples:**
- `amt` must be ordinary numeric text with at most two decimal places (`AMOUNT_PATTERN = r"^-?[0-9]+(\.[0-9]{1,2})?$"`) and parse under `try_cast(amt AS DECIMAL(18, 2))`.
- **Merged validation reason**: Both malformed/out-of-range text (e.g., `"abc"`) and values with more than two decimal places that `try_cast` would silently round (e.g., `"19.999"` → `20.00`) now produce one clear reason: `invalid_amount` (replacing the legacy `unparseable_or_out_of_range_amount` and `amount_precision_changed` reasons). Non-ordinary formats like `"+5"`, `".50"`, or `"1e3"` are also rejected as `invalid_amount`.
- **Non-positive check**: Any parsed `amount <= 0` (such as `"0.00"` or `"-15.25"`) records `non_positive_amount`.

In [ ]:
def clean_and_check_fields(bronze_df):
    df = (
        bronze_df
        .withColumn("raw_trans_num", F.col("trans_num"))
        .withColumn("raw_cc_num", F.col("cc_num"))
        .withColumn("raw_merchant", F.col("merchant"))
        .withColumn("raw_amt", F.col("amt"))
        .withColumn("raw_trans_date_trans_time", F.col("trans_date_trans_time"))
    )

    # Trim spaces and turn empty text into NULL across all 21 business columns.
    for column_name in BUSINESS_COLUMNS:
        df = df.withColumn(column_name, F.nullif(F.trim(F.col(column_name)), F.lit("")))

    # try_cast returns NULL on bad text instead of raising an ANSI cast error.
    df = (
        df
        .withColumn("event_time", F.expr("try_cast(trans_date_trans_time AS TIMESTAMP_NTZ)"))
        .withColumn("amount", F.expr("try_cast(amt AS DECIMAL(18, 2))"))
    )

    # Each F.when(...) returns a reason string when a check fails, otherwise NULL.
    # F.array_compact removes NULLs so only actual failure reasons remain.
    return df.withColumn(
        "field_reasons",
        F.array_compact(F.array(
            F.when(F.col("trans_num").isNull(), "missing_transaction_id"),
            F.when(F.col("cc_num").isNull(), "missing_card_id"),
            F.when(F.col("merchant").isNull(), "missing_merchant"),
            F.when(F.col("trans_date_trans_time").isNull(), "missing_event_time"),
            F.when(
                F.col("trans_date_trans_time").isNotNull() & F.col("event_time").isNull(),
                "unparseable_event_time",
            ),
            F.when(F.col("amt").isNull(), "missing_amount"),
            F.when(
                F.col("amt").isNotNull()
                & (~F.col("amt").rlike(AMOUNT_PATTERN) | F.col("amount").isNull()),
                "invalid_amount",
            ),
            F.when(F.col("amount") <= 0, "non_positive_amount"),
            F.when(
                F.col("_rescued_data").isNotNull() & (F.trim(F.col("_rescued_data")) != ""),
                "unexpected_rescued_data",
            ),
        )),
    )

## Step 2: Find duplicate transaction IDs

For each `trans_num`, we use window functions (`Window.partitionBy("trans_num")`) across all Bronze rows (including rows with field errors so an invalid copy never hides a conflict):
- `id_copies`: how many rows share the same `trans_num`.
- `id_versions`: how many *distinct* business versions exist for that `trans_num`. `F.collect_set(F.struct(*BUSINESS_COLUMNS)).over(same_id)` collects unique 21-column structs and handles `NULL` fields natively without JSON serialization.
- `duplicate_rank`: `1, 2, 3...` in deterministic arrival order (`_ingestion_timestamp`, `_ingested_file_path`, numeric `source_row_index`).

| `duplicate_status` | When |
| --- | --- |
| `missing_id` | `trans_num` is `NULL` |
| `unique_id` | only one row has this `trans_num` (`id_copies == 1`) |
| `conflicting_duplicate` | multiple rows share this `trans_num` with different business values (`id_versions > 1`) |
| `identical_duplicate_primary` | multiple identical rows (`id_versions == 1`), first copy (`duplicate_rank == 1`) |
| `identical_duplicate_extra` | multiple identical rows (`id_versions == 1`), later copy (`duplicate_rank > 1`) |

In [ ]:
def find_duplicates(df):
    same_id = Window.partitionBy("trans_num")
    arrival_order = same_id.orderBy(
        "_ingestion_timestamp",
        "_ingested_file_path",
        F.expr("try_cast(source_row_index AS BIGINT)"),  # numeric order: 9 comes before 10
        "source_row_index",
    )

    return (
        df
        .withColumn("id_copies", F.count("*").over(same_id))
        .withColumn("id_versions", F.size(F.collect_set(F.struct(*BUSINESS_COLUMNS)).over(same_id)))
        .withColumn("duplicate_rank", F.when(F.col("trans_num").isNotNull(), F.row_number().over(arrival_order)))
        .withColumn(
            "duplicate_status",
            F.when(F.col("trans_num").isNull(), "missing_id")
            .when(F.col("id_copies") == 1, "unique_id")
            .when(F.col("id_versions") > 1, "conflicting_duplicate")
            .when(F.col("duplicate_rank") == 1, "identical_duplicate_primary")
            .otherwise("identical_duplicate_extra"),
        )
    )

## Step 3: Decide where each row goes

`validation_reasons` combines `field_reasons` and `duplicate_reasons` (`F.concat`). Then:
- **No reasons (`size(validation_reasons) == 0`)** → `accepted`
- **Extra identical copy of a valid row (`identical_duplicate_extra` and `size(field_reasons) == 0`)** → `duplicate_identical`
- **Anything else** → `quarantined`

In [ ]:
def route_rows(df):
    return (
        df
        .withColumn(
            "duplicate_reasons",
            F.array_compact(F.array(
                F.when(F.col("duplicate_status") == "conflicting_duplicate", "conflicting_duplicate_transaction_id"),
                F.when(F.col("duplicate_status") == "identical_duplicate_extra", "duplicate_identical_transaction_id"),
            )),
        )
        .withColumn("validation_reasons", F.concat("field_reasons", "duplicate_reasons"))
        .withColumn(
            "record_disposition",
            F.when(F.size("validation_reasons") == 0, "accepted")
            .when(
                (F.col("duplicate_status") == "identical_duplicate_extra") & (F.size("field_reasons") == 0),
                "duplicate_identical",
            )
            .otherwise("quarantined"),
        )
    )


def classify_transactions(bronze_df):
    return route_rows(find_duplicates(clean_and_check_fields(bronze_df)))

## Self-test on a small hand-made fixture

Before running on real Bronze data, we test `classify_transactions` on 19 rows covering:
- Valid row (`1`), required-field checks (`2`, `7`, `17`), malformed/3-decimal/non-positive amounts (`3`, `4`, `5`), bad timestamp (`6`), and rescued data (`8`).
- Identical duplicates (`9` and `10`, verifying numeric row-index order `9` before `10`), conflicting duplicates (`11` and `12`), and `NULL` handling (`13`–`16`).
- Multiple errors on a single row (`17`: missing merchant, event time, and amount together).
- Conflicting duplicates where one occurrence is invalid (`18` and `19`: valid `amt="40.00"` vs. invalid `amt="corrupted"`, proving both are quarantined).

In [ ]:
fixture_schema = (
    ", ".join(f"`{name}` string" for name in BUSINESS_COLUMNS)  # backticks: some names are SQL keywords
    + ", replay_batch date, source_file string, source_row_index string"
    + ", _ingested_file_path string, _ingestion_timestamp timestamp, _rescued_data string"
)


def make_row(row_index, **changes):
    row = {name: None for name in BUSINESS_COLUMNS}
    row.update(
        trans_num=f"T{row_index}", cc_num="1111", merchant="Shop", category="grocery",
        amt="10.00", trans_date_trans_time="2019-01-01 10:00:00",
        replay_batch=date(2019, 1, 1), source_file="fixture.csv", source_row_index=str(row_index),
        _ingested_file_path="fixture.parquet", _ingestion_timestamp=datetime(2019, 1, 2, 0, 0, 0),
        _rescued_data=None,
    )
    row.update(changes)
    return row


fixture_rows = [
    make_row(1),                                                                        # valid
    make_row(2, trans_num="   "),                                                       # blank transaction ID
    make_row(3, amt="abc"),                                                             # non-numeric amount
    make_row(4, amt="19.999"),                                                          # >2 decimal places
    make_row(5, amt="0.00"),                                                            # non-positive amount
    make_row(6, trans_date_trans_time="not a date"),                                    # unparseable timestamp
    make_row(7, cc_num=None),                                                           # missing card ID
    make_row(8, _rescued_data='{"extra": 1}'),                                          # unexpected rescued data
    make_row(9, trans_num="SAME"),                                                      # identical copies: 9 first
    make_row(10, trans_num="SAME"),
    make_row(11, trans_num="CONFLICT", amt="10.00"),                                    # same ID, different amount
    make_row(12, trans_num="CONFLICT", amt="99.00"),
    make_row(13, trans_num="NULLS", category=None),                                     # identical copies with NULLs
    make_row(14, trans_num="NULLS", category=None),
    make_row(15, trans_num="NULLS2", category=None),                                    # NULL category vs real category
    make_row(16, trans_num="NULLS2", category="grocery"),                               # = conflicting, not identical
    make_row(17, merchant="   ", trans_date_trans_time=None, amt=""),                   # multiple missing required fields
    make_row(18, trans_num="CONFLICT_BAD", amt="40.00"),                                # conflicting duplicate with
    make_row(19, trans_num="CONFLICT_BAD", amt="corrupted"),                            # an invalid occurrence
]

fixture_result = classify_transactions(spark.createDataFrame(fixture_rows, schema=fixture_schema))
collected = fixture_result.collect()
actual = {r["source_row_index"]: r["record_disposition"] for r in collected}

expected = {
    "1": "accepted",
    "2": "quarantined", "3": "quarantined", "4": "quarantined", "5": "quarantined",
    "6": "quarantined", "7": "quarantined", "8": "quarantined",
    "9": "accepted", "10": "duplicate_identical",
    "11": "quarantined", "12": "quarantined",
    "13": "accepted", "14": "duplicate_identical",
    "15": "quarantined", "16": "quarantined",
    "17": "quarantined", "18": "quarantined", "19": "quarantined",
}
assert actual == expected, f"Self-test failed.\nExpected: {expected}\nActual:   {actual}"

reasons = {r["source_row_index"]: r["validation_reasons"] for r in collected}
assert reasons["2"] == ["missing_transaction_id"]
assert reasons["3"] == ["invalid_amount"] and reasons["4"] == ["invalid_amount"]
assert reasons["5"] == ["non_positive_amount"]
assert reasons["6"] == ["unparseable_event_time"]
assert reasons["7"] == ["missing_card_id"]
assert reasons["8"] == ["unexpected_rescued_data"]
assert reasons["10"] == ["duplicate_identical_transaction_id"]
assert reasons["11"] == ["conflicting_duplicate_transaction_id"]
assert reasons["17"] == ["missing_merchant", "missing_event_time", "missing_amount"]
assert reasons["18"] == ["conflicting_duplicate_transaction_id"]
assert reasons["19"] == ["invalid_amount", "conflicting_duplicate_transaction_id"]

print("Self-test passed: 19 of 19 rows routed and explained as expected.")

## Run on real Bronze data

In [ ]:
bronze_df = spark.table(BRONZE_TABLE)
assert "is_fraud" not in bronze_df.columns, "Fraud labels must not appear in Bronze"

bronze_rows = bronze_df.count()
classified = classify_transactions(bronze_df)

accepted_df = classified.filter("record_disposition = 'accepted'").select(*ACCEPTED_COLUMNS)
quarantine_df = classified.filter("record_disposition != 'accepted'").select(*QUARANTINE_COLUMNS)

# Row accounting: every Bronze row must land in exactly one disposition.
counts = {r["record_disposition"]: r["count"] for r in classified.groupBy("record_disposition").count().collect()}
assert sum(counts.values()) == bronze_rows, "Some Bronze rows were lost or duplicated"
print(f"{bronze_rows:,} Bronze rows -> {counts}")

## Save Silver and quarantine

Both tables are **rebuilt from all of Bronze** (`overwrite`). A later batch can add a conflicting copy of an
old ID, and rebuilding is the simple way to move the earlier copy to quarantine.
Run this cell and the verification cell below together.

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")

accepted_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(SILVER_TRANSACTIONS_TABLE)
quarantine_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(SILVER_QUARANTINE_TABLE)

print(f"Saved {SILVER_TRANSACTIONS_TABLE} and {SILVER_QUARANTINE_TABLE}.")

## Verify the saved tables

In [ ]:
silver = spark.table(SILVER_TRANSACTIONS_TABLE)
quarantine = spark.table(SILVER_QUARANTINE_TABLE)

silver_rows = silver.count()
quarantine_rows = quarantine.count()

# 1. Nothing lost, no fraud labels, no personal columns.
assert silver_rows + quarantine_rows == bronze_rows, "Silver + quarantine must equal Bronze"
assert silver.columns == ACCEPTED_COLUMNS and quarantine.columns == QUARANTINE_COLUMNS
assert "is_fraud" not in silver.columns and "is_fraud" not in quarantine.columns

# 2. Accepted IDs are never NULL and never repeated.
assert silver.filter(F.col("trans_num").isNull()).count() == 0, "NULL trans_num in Silver"
assert silver.select("trans_num").distinct().count() == silver_rows, "Duplicate trans_num in Silver"

# 3. Each ingested replay day matches the manifest (expected rows per day).
saved_per_day = (
    silver.select("replay_batch").unionByName(quarantine.select("replay_batch"))
    .groupBy("replay_batch").agg(F.count("*").alias("saved_rows"))
)
manifest = spark.read.parquet(MANIFEST_PATH)
wrong_days = manifest.join(saved_per_day, "replay_batch").filter(F.col("saved_rows") != F.col("expected_transaction_count"))
assert wrong_days.count() == 0, "Some replay days do not match the manifest"

print(f"Verified: {silver_rows:,} accepted + {quarantine_rows:,} quarantined = {bronze_rows:,} Bronze rows.")
display(manifest.join(saved_per_day, "replay_batch").orderBy("replay_batch"))